<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    3.lab.2. Auto MPG Regression - Neural Networks
  </strong>
</div>

In [ ]:
# ⚠️ IMPORTANT NOTICE FOR STUDENTS ⚠️
#
# Please make sure to check the official instructions for this assignment in Canvas LMS
# as they may have been updated or changed. The instructions above are provided for
# reference only and may not reflect the most current requirements.
#
# Always refer to Canvas LMS for:
# - Latest assignment requirements
# - Due dates
# - Grading criteria
# - Any special instructions
#
# When in doubt, ask your instructor for clarification.

from seaborn import load_dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
df = load_dataset("mpg")
df.head()

In [ ]:
print("Información del dataset:")
print(f"Forma: {df.shape}")
print("\nPrimeras 5 filas:")
print(df.head())

In [ ]:
print("\nValores nulos en el dataset:")
print(df.isnull().sum())

print("\nEstadísticas descriptivas:")
print(df.describe())

In [ ]:
df = df.dropna()
print(f"\nTamaño del dataset después de eliminar valores nulos: {df.shape}")

df["origin"] = df["origin"].astype("category")
df = pd.get_dummies(df, columns=["origin"], drop_first=True)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df["mpg"], kde=True)
plt.title("Distribución de MPG")
plt.xlabel("MPG (millas por galón)")
plt.ylabel("Frecuencia")
plt.grid(True)
plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=["float64", "int64"])

corr_matrix = numeric_df.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de Correlación")
plt.tight_layout()
plt.show()

In [ ]:
X = df.drop("mpg", axis=1)
y = df["mpg"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTamaño del conjunto de entrenamiento: {X_train.shape}")
print(f"Tamaño del conjunto de prueba: {X_test.shape}")

In [ ]:
X_train_numeric = X_train.select_dtypes(include=["float64", "int64"])
X_test_numeric = X_test.select_dtypes(include=["float64", "int64"])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_numeric)
X_test_scaled = scaler.transform(X_test_numeric)

X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

In [ ]:
class MPGDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_dataset = MPGDataset(X_train_tensor, y_train_tensor)
test_dataset = MPGDataset(X_test_tensor, y_test_tensor)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, input_size):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
class DeepNN(nn.Module):
    def __init__(self, input_size):
        super(DeepNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [ ]:
class WideNN(nn.Module):
    def __init__(self, input_size):
        super(WideNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
class RegularizedNN(nn.Module):
    def __init__(self, input_size):
        super(RegularizedNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout1(x)
        x = self.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

In [ ]:
def train_and_evaluate(
    model, train_loader, test_loader, criterion, optimizer, num_epochs=100
):
    train_losses = []
    test_losses = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)

        epoch_train_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_train_loss)

        model.eval()
        running_loss = 0.0
        with torch.no_grad():
            for inputs, targets in test_loader:
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                running_loss += loss.item() * inputs.size(0)

        epoch_test_loss = running_loss / len(test_loader.dataset)
        test_losses.append(epoch_test_loss)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(
                f"Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_train_loss:.4f}, Test Loss: {epoch_test_loss:.4f}"
            )

    return train_losses, test_losses

In [ ]:
def evaluate_model(model, test_loader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for inputs, targets in test_loader:
            outputs = model(inputs)
            y_true.extend(targets.numpy().flatten())
            y_pred.extend(outputs.numpy().flatten())

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    return mse, rmse, r2, y_true, y_pred

In [ ]:
input_size = X_train_tensor.shape[1]
criterion = nn.MSELoss()
num_epochs = 100
learning_rate = 0.001

In [ ]:
simple_model = SimpleNN(input_size)
optimizer = optim.Adam(simple_model.parameters(), lr=learning_rate)
simple_train_losses, simple_test_losses = train_and_evaluate(
    simple_model, train_loader, test_loader, criterion, optimizer, num_epochs
)
simple_mse, simple_rmse, simple_r2, simple_y_true, simple_y_pred = evaluate_model(
    simple_model, test_loader
)

In [ ]:
deep_model = DeepNN(input_size)
optimizer = optim.Adam(deep_model.parameters(), lr=learning_rate)
deep_train_losses, deep_test_losses = train_and_evaluate(
    deep_model, train_loader, test_loader, criterion, optimizer, num_epochs
)
deep_mse, deep_rmse, deep_r2, deep_y_true, deep_y_pred = evaluate_model(
    deep_model, test_loader
)

In [ ]:
wide_model = WideNN(input_size)
optimizer = optim.Adam(wide_model.parameters(), lr=learning_rate)
wide_train_losses, wide_test_losses = train_and_evaluate(
    wide_model, train_loader, test_loader, criterion, optimizer, num_epochs
)
wide_mse, wide_rmse, wide_r2, wide_y_true, wide_y_pred = evaluate_model(
    wide_model, test_loader
)

In [ ]:
reg_model = RegularizedNN(input_size)
optimizer = optim.Adam(reg_model.parameters(), lr=learning_rate, weight_decay=0.001)
reg_train_losses, reg_test_losses = train_and_evaluate(
    reg_model, train_loader, test_loader, criterion, optimizer, num_epochs
)
reg_mse, reg_rmse, reg_r2, reg_y_true, reg_y_pred = evaluate_model(
    reg_model, test_loader
)

In [ ]:
plt.figure(figsize=(14, 7))

plt.subplot(1, 2, 1)
plt.plot(simple_train_losses, label="Simple - Train")
plt.plot(simple_test_losses, label="Simple - Test")
plt.plot(deep_train_losses, label="Deep - Train")
plt.plot(deep_test_losses, label="Deep - Test")
plt.plot(wide_train_losses, label="Wide - Train")
plt.plot(wide_test_losses, label="Wide - Test")
plt.plot(reg_train_losses, label="Regularized - Train")
plt.plot(reg_test_losses, label="Regularized - Test")
plt.xlabel("Época")
plt.ylabel("Pérdida (MSE)")
plt.title("Curvas de Aprendizaje")
plt.legend()
plt.grid(True)

In [ ]:
plt.subplot(1, 2, 2)
plt.plot(simple_test_losses, label="Simple NN")
plt.plot(deep_test_losses, label="Deep NN")
plt.plot(wide_test_losses, label="Wide NN")
plt.plot(reg_test_losses, label="Regularized NN")
plt.xlabel("Época")
plt.ylabel("Pérdida de Test (MSE)")
plt.title("Comparación de Pérdida en Test")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))

plt.subplot(2, 2, 1)
plt.scatter(simple_y_true, simple_y_pred, alpha=0.5)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], "k--")
plt.xlabel("MPG Real")
plt.ylabel("MPG Predicho")
plt.title(f"Simple NN (R² = {simple_r2:.3f})")
plt.grid(True)

In [ ]:
plt.subplot(2, 2, 2)
plt.scatter(deep_y_true, deep_y_pred, alpha=0.5)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], "k--")
plt.xlabel("MPG Real")
plt.ylabel("MPG Predicho")
plt.title(f"Deep NN (R² = {deep_r2:.3f})")
plt.grid(True)

In [ ]:
plt.subplot(2, 2, 3)
plt.scatter(wide_y_true, wide_y_pred, alpha=0.5)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], "k--")
plt.xlabel("MPG Real")
plt.ylabel("MPG Predicho")
plt.title(f"Wide NN (R² = {wide_r2:.3f})")
plt.grid(True)

In [ ]:
plt.subplot(2, 2, 4)
plt.scatter(reg_y_true, reg_y_pred, alpha=0.5)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], "k--")
plt.xlabel("MPG Real")
plt.ylabel("MPG Predicho")
plt.title(f"Regularized NN (R² = {reg_r2:.3f})")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
results = {
    "Modelo": ["Simple NN", "Deep NN", "Wide NN", "Regularized NN"],
    "MSE": [simple_mse, deep_mse, wide_mse, reg_mse],
    "RMSE": [simple_rmse, deep_rmse, wide_rmse, reg_rmse],
    "R²": [simple_r2, deep_r2, wide_r2, reg_r2],
}

results_df = pd.DataFrame(results)
print("\nComparación de Modelos:")
print(results_df)

In [ ]:
features = X_train_numeric.columns

weights = simple_model.fc1.weight.data.numpy()

feature_importance = np.abs(weights).mean(axis=0)

if len(features) != len(feature_importance):
    raise ValueError(
        f"El número de características ({len(features)}) no coincide con el número de pesos ({len(feature_importance)})."
    )

plt.figure(figsize=(10, 6))
plt.bar(features, feature_importance)
plt.xlabel("Características")
plt.ylabel("Importancia Relativa")
plt.title("Importancia de Características en el Modelo Simple")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
best_model_idx = results_df["R²"].idxmax()
best_model_name = results_df.loc[best_model_idx, "Modelo"]
best_y_true = None
best_y_pred = None

if best_model_name == "Simple NN":
    best_y_true, best_y_pred = simple_y_true, simple_y_pred
elif best_model_name == "Deep NN":
    best_y_true, best_y_pred = deep_y_true, deep_y_pred
elif best_model_name == "Wide NN":
    best_y_true, best_y_pred = wide_y_true, wide_y_pred
else:
    best_y_true, best_y_pred = reg_y_true, reg_y_pred

residuos = np.array(best_y_true) - np.array(best_y_pred)


In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(best_y_pred, residuos)
plt.axhline(y=0, color="r", linestyle="-")
plt.xlabel("Valores Predichos")
plt.ylabel("Residuos")
plt.title("Residuos vs Predicciones")
plt.grid(True)

In [ ]:
plt.subplot(1, 2, 2)
plt.hist(residuos, bins=20)
plt.xlabel("Residuos")
plt.ylabel("Frecuencia")
plt.title("Distribución de Residuos")
plt.grid(True)

plt.tight_layout()
plt.show()

Comparacion del Rendimiento de los Modelos:

    La Red Neuronal Profunda (Deep NN) obtuvo el mejor desempeño con un R² de 0.889, lo que indica que explico aproximadamente el 89% de la varianza en MPG.
    La Red Neuronal Regularizada quedo en segundo lugar con un R² de 0.872.
    La Red Neuronal Ancha (Wide NN) alcanzo un R² de 0.856.
    La Red Neuronal Simple tuvo el rendimiento mas bajo con un R² de 0.792.

Dinamica de Aprendizaje:

    Las curvas de aprendizaje mostraron que la Deep NN convergio mas rápidamente a un error menor.
    La Red Neuronal Regularizada mostro una notable diferencia entre la perdida de entrenamiento y de prueba, lo que sugiere que evito eficazmente el sobreajuste.
    Todos los modelos mostraron una mejora significativa durante el período de entrenamiento, con perdidas de prueba disminuyendo a lo largo de las 100 epocas.

Precision de Prediccion:

    La Deep NN tuvo el RMSE mas bajo (2.38), lo que significa que sus predicciones fueron las mas cercanas a los valores reales de MPG.
    Los graficos de dispersion de valores predichos vs. reales mostraron que las predicciones se alineaban bien con los valores reales, con puntos agrupados cerca de la línea diagonal ideal.

Impacto de la Arquitectura del Modelo:

    Agregar profundidad (mas capas) resulto mas beneficioso que agregar anchura (mas neuronas en una sola capa).
    Las técnicas de regularizacion (dropout y weight decay) mejoraron eficazmente el rendimiento de generalizacion en comparacion con el modelo simple.